# CRISPR Guide Enumeration

![CRISPR Guide Enumeration](https://proto-bio.github.io/proto-assets/images/tool/crispr_guides/hero.png)

This notebook demonstrates `run_crispr_guides_enumerate`, which scans reference-oriented DNA intervals for every protospacer sitting next to a PAM. It is deterministic string matching: no activity model, no off-target search, and no candidate selection, so the result is the full auditable pool you rank with other tools.

In [ ]:
from proto_tools.utils.notebook_docs import display_api_reference
from proto_tools.tools.gene_annotation.crispr_guides import (
    CRISPRGuideEnumerationConfig,
    CRISPRGuideEnumerationInput,
    CRISPRGuideRegion,
    run_crispr_guides_enumerate,
)

## API Reference

In [ ]:
display_api_reference("crispr-guides-enumerate", "input", "run_crispr_guides_enumerate")

In [ ]:
display_api_reference("crispr-guides-enumerate", "config", "run_crispr_guides_enumerate")

In [ ]:
display_api_reference("crispr-guides-enumerate", "output", "run_crispr_guides_enumerate")

## Enumerate SpCas9 Guides

Each region is supplied in reference plus-strand orientation with the 1-indexed coordinate of its first base. Every returned coordinate is derived from that, so plus- and minus-strand candidates are reported on the same reference axis.

In [ ]:
# A synthetic interval with one 20-nt SpCas9 protospacer followed by TGG.
region = CRISPRGuideRegion(
    sequence_id="synthetic_interval",
    sequence="AAAAGACTGCATTGCATGACTGCATGGAAA",
    start=1,
    reference_id="synthetic-v1",
)
result = run_crispr_guides_enumerate(
    CRISPRGuideEnumerationInput(regions=region),
    CRISPRGuideEnumerationConfig(strands="plus"),
)
[(candidate.protospacer, candidate.pam, candidate.pam_strand) for candidate in result.results[0].candidates]

## Feeding an Activity Model

The default 4-nt and 3-nt exterior flanks make `context_sequence` the 30-nt window Rule Set 3 scores: 4-nt flank, 20-nt protospacer, 3-nt PAM, 3-nt flank. A candidate close to an interval edge keeps only the flank the interval provides, so check the length before passing it on.

In [ ]:
candidate = result.results[0].candidates[0]
print(candidate.context_sequence, len(candidate.context_sequence))
print(f"protospacer {candidate.protospacer_start}-{candidate.protospacer_end} on {candidate.pam_strand}")

## Other Nucleases

The PAM is written in nuclease orientation as an IUPAC motif, so a five-prime PAM such as Cas12a `TTTV` only needs `pam_side="five_prime"`. Scanning `both` strands reports candidates from each reference strand against the same coordinates.

In [ ]:
cas12a = run_crispr_guides_enumerate(
    CRISPRGuideEnumerationInput(
        regions=CRISPRGuideRegion(
            sequence_id="synthetic_cas12a",
            sequence="TTTAGACTGCATTGCATGACTGCA",
        )
    ),
    CRISPRGuideEnumerationConfig(
        pam="TTTV", pam_side="five_prime", strands="both", context_5p=0, context_3p=0
    ),
)
cas12a.results[0].candidates

## Export Results

JSON keeps the per-region nesting; CSV flattens to one row per candidate, with its region's `sequence_id` in the first column.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="crispr_guide_candidates", export_path=output_dir, file_format="json")
print(f"Guide candidates exported to {output_dir}")